# AML - LEAVE ONE VARIANT OUT - SUBSET EXOME FILES - UVA 
This script creates a PLINK BED file from the exome dataset by subsetting variants that pass quality control within the genes of interest. The script must be submitted once for each chromosome containing a gene of interest. The resulting BED file is then used in a leave-one-variant-out analysis to determine which variants contribute most to the genome-wide significance of the genes of interest. Additionally, the script requires a list of variants that passed exome quality control and are located within the genes of interest.

In [ ]:
# Load packages 
import os
import pandas as pd 

In [ ]:
# Set bucket environment
my_bucket = os.getenv('WORKSPACE_BUCKET')

# COMMAND 

In [ ]:
%%writefile lovo_exome_subset.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
        # Check if the cohort file for the current population exists
        if [ ! -f "${cohort}/${pop}-like_exome_qc.txt" ]; then
            echo "Cohort file for ${pop} not found!"
            exit 1
        fi

        # Copy new plink module into the plink2 
        cp "${plink2}" plink2

        # Grant permision to run plink2 in environment
        chmod +x plink2

        # Run PLINK2 for the current population
        ./plink2 \
          --bfile "${genos}/exome.chr${chrom}" \
          --keep "${cohort}/${pop}-like_exome_qc.txt" \
          --extract "${variants}" \
          --make-bed \
          --out "${output}/${pop}_LOVO_exome_chr${chrom}"
done

In [ ]:
!gsutil cp lovo_exome_subset.sh {my_bucket}/data/scripts/

In [ ]:
!gsutil cat {my_bucket}/data/scripts/lovo_exome_subset.sh

# SUBMIT COMMAND 

In [ ]:
%%bash --out lovo_exome_subset

#############################################################################
# DSUB: SUBMIT SCRIPT FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Source the dsub setup script, which likely contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=11
UPPER=12

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

#############################################################################
# SUBMIT DSUB SCRIPT JOB FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Use a prebuilt container with PLINK2
# Set total disk space required 
# Define logging directory for dsub execution logs
# Input PLINK files (BED, BIM, FAM) from Google Cloud Storage (GCS)
# Provide cohort sample lists (recursively input from GCS)
# Define output directory for variant QC results (recursively stores outputs)
# Set environment variables for the script
# Specify the script to execute within the container
  
aou_dsub \
  --disk-size 300 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input variants="${WORKSPACE_BUCKET}/data/annotation_table/EUR_LOVO_annotation/EUR_LOVO_annotation_chr${chromo}/LOVO-annotation-chr${chromo}.txt" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/lovo_exome_subset.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
%%bash --out lovo_exome_subset

#############################################################################
# DSUB: SUBMIT SCRIPT FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Source the dsub setup script, which likely contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=19
UPPER=20

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

#############################################################################
# SUBMIT DSUB SCRIPT JOB FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Use a prebuilt container with PLINK2
# Set total disk space required 
# Define logging directory for dsub execution logs
# Input PLINK files (BED, BIM, FAM) from Google Cloud Storage (GCS)
# Provide cohort sample lists (recursively input from GCS)
# Define output directory for variant QC results (recursively stores outputs)
# Set environment variables for the script
# Specify the script to execute within the container
  
aou_dsub \
  --disk-size 300 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input variants="${WORKSPACE_BUCKET}/data/annotation_table/EUR_LOVO_annotation/EUR_LOVO_annotation_chr${chromo}/LOVO-annotation-chr${chromo}.txt" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/lovo_exome_subset.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done

In [ ]:
%%bash --out lovo_exome_subset

#############################################################################
# DSUB: SUBMIT SCRIPT FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Source the dsub setup script, which likely contains environment setup and utility functions
source ~/aou_dsub.bash 

# Define chromosome range for processing
LOWER=21
UPPER=22

# Loop through the specified chromosome range
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

#############################################################################
# SUBMIT DSUB SCRIPT JOB FOR VARIANT SUMMARY STATISTICS 
#############################################################################

# Use a prebuilt container with PLINK2
# Set total disk space required 
# Define logging directory for dsub execution logs
# Input PLINK files (BED, BIM, FAM) from Google Cloud Storage (GCS)
# Provide cohort sample lists (recursively input from GCS)
# Define output directory for variant QC results (recursively stores outputs)
# Set environment variables for the script
# Specify the script to execute within the container
  
aou_dsub \
  --disk-size 100 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input variants="${WORKSPACE_BUCKET}/data/annotation_table/EUR_LOVO_annotation/EUR_LOVO_annotation_chr${chromo}/LOVO-annotation-chr${chromo}.txt" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/" \
  --env chrom=${chromo} \
  --env genos="${mnt_path}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/lovo_exome_subset.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################

done